In [1]:
%cd ../../..

/home/hoanghu/projects/Food-Waste-Optimization


In [10]:
import joblib
from itertools import permutations

import numpy as np
import pandas as pd
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBRegressor

# Read data, trained models

In [3]:
path = "trained_models/pos/phase_4/xgb_cat_Nov13.json"

reg = XGBRegressor(tree_method="hist", enable_categorical=True)
reg.load_model(path)

In [5]:
path = "trained_models/encoder/phase_4/targetenc_meal_id_Nov13.pkl"
enc_meal_id = joblib.load(path)

In [7]:
path = "data/processed/phase_4/dim_meals.xlsx"

dim_meals = pd.read_excel(path)
dim_meals.head()

,meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
0,9017,vegan,24-25,True,True,che-exa,vegan-miscellaneous,NaN
1,7201,vegan,23-24,False,False,NaN,NaN,NaN
2,9032,vegan,23-24,False,False,NaN,NaN,NaN
3,9102,vegan,23-24,False,False,NaN,NaN,NaN
4,7010,vegetarian,24-25,False,False,che-exa,NaN,71.0


In [18]:
meal_info = dim_meals[['meal_id', 'meal_type_1', 'pcs_mean']]
meal_info.columns = ['id', 'type', 'mean']

# Forecast

In [8]:
meal_ids = [7614, 9500047, 1307]
restaurant = "che"
date = pd.to_datetime("2024-12-01")

In [19]:
feat = dim_meals[dim_meals['meal_id'].isin(meal_ids)][['meal_id', 'meal_type_1']].copy()
feat.columns = ['meal_id', 'meal_type']

feat['meal_type'] = feat['meal_type'].map({
    'meat':    1, # 'meat',
    'fish':    2, # 'fish',
    'vegan':    3, # 'vegan',
    'vegetarian':    4, # 'vegetarian',
    'chicken':    5, # 'chicken'
})


feat['restaurant'] = restaurant

THETA = 5
records = []
for r in feat.itertuples():
    ids = set(meal_ids)
    ids.remove(r.meal_id)
    
    for tup in permutations(ids):
        tup = [*tup]

        # pad
        if len(tup) < THETA - 1:
            tup.extend([0]*(THETA - 1 - len(tup)))
        
        records.append({
            'date': date,
            'restaurant': r.restaurant,
            'meal_id': r.meal_id,
            'meal_id_other1': tup[0],
            'meal_id_other2': tup[1],
            'meal_id_other3': tup[2],
            'meal_id_other4': tup[3],
        })

feat = pd.DataFrame.from_records(records)



def _encode_date_cyclic(t, period_week: int = 7, period_day: int = 31, period_month: int = 12):
    def get_sin_encoding(x, period: int):
        return np.sin(2 * np.pi * x / period)
    def get_cos_encoding(x, period: int):
        return np.cos(2 * np.pi * x / period)  

    return pd.Series({
        'weekday_sin': get_sin_encoding(t.weekday(), period_week),
        'weekday_cos': get_cos_encoding(t.weekday(), period_week),
        'day_sin': get_sin_encoding(t.day, period_day),
        'day_cos': get_cos_encoding(t.day, period_day),
        'month_sin': get_sin_encoding(t.month, period_month),
        'month_cos': get_cos_encoding(t.month, period_month),
    })

datetime_encoded = feat['date'].apply(_encode_date_cyclic)
feat = pd.concat([feat, datetime_encoded], axis=1)


feat = (
    feat
    .merge(meal_info, how='left', left_on=['meal_id'], right_on=['id'])
    .drop(columns='id')
    .rename(columns={'type': 'meal_type', 'mean': 'pcs_mean'})

    .merge(meal_info, how='left', left_on=['meal_id_other1'], right_on=['id'])
    .drop(columns='id')
    .rename(columns={'type': 'meal_type_other1', 'mean': 'pcs_mean_other1'})

    .merge(meal_info, how='left', left_on=['meal_id_other2'], right_on=['id'])
    .drop(columns='id')
    .rename(columns={'type': 'meal_type_other2', 'mean': 'pcs_mean_other2'})

    .merge(meal_info, how='left', left_on=['meal_id_other3'], right_on=['id'])
    .drop(columns='id')
    .rename(columns={'type': 'meal_type_other3', 'mean': 'pcs_mean_other3'})

    .merge(meal_info, how='left', left_on=['meal_id_other4'], right_on=['id'])
    .drop(columns='id')
    .rename(columns={'type': 'meal_type_other4', 'mean': 'pcs_mean_other4'})
)
feat = feat[~feat['pcs_mean'].isna()].fillna(0)         # Filter out records having no `pcs_mean` and impute data


# Encode meal_id

feat['meal_id_enc'] = enc_meal_id.transform(feat[['meal_id']])

cols = ['meal_id_other1', 'meal_id_other2', 'meal_id_other3', 'meal_id_other4']
for col in cols:
    encoded = enc_meal_id.transform(feat[[col]])
    mask = (feat[col] != 0).astype(np.int32)
    encoded = encoded.squeeze() * mask

    feat[f'{col}_enc'] = encoded


# Assign categorical column type
cols_cat = [
    'restaurant',

    'meal_type',


    'meal_type_other1',
    'meal_type_other2',
    'meal_type_other3',
    'meal_type_other4',
]
for col in cols_cat:
    feat[col] = feat[col].astype('category')



# Keep important columns
cols_X = [
    'weekday_sin',
    'weekday_cos',
    'day_sin',
    'day_cos',
    'month_sin',
    'month_cos',

    'restaurant',

    'meal_id_enc',
    'meal_type',

    'pcs_mean',


    'meal_id_other1_enc',
    'meal_id_other2_enc',
    'meal_id_other3_enc',
    'meal_id_other4_enc',

    'meal_type_other1',
    'meal_type_other2',
    'meal_type_other3',
    'meal_type_other4',

    'pcs_mean_other1',
    'pcs_mean_other2',
    'pcs_mean_other3',
    'pcs_mean_other4',
]
X = feat[cols_X]

X.head()

/tmp/ipykernel_13629/941296666.py:83: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  feat = feat[~feat['pcs_mean'].isna()].fillna(0)         # Filter out records having no `pcs_mean` and impute data
/home/hoanghu/projects/Food-Waste-Optimization/.venv/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but TargetEncoder was fitted without feature names
  warnings.warn(
/home/hoanghu/projects/Food-Waste-Optimization/.venv/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but TargetEncoder was fitted without feature names
  warnings.warn(
/home/hoanghu/projects/Food-Waste-Optimization/.venv/lib/python3.10/site-packages/sklearn/base.py:486: UserWarning: X has feature names, but TargetEncoder was fitted withou

,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos,restaurant,meal_id_enc,meal_type,pcs_mean,meal_id_other1_enc,meal_id_other2_enc,meal_id_other3_enc,meal_id_other4_enc,meal_type_other1,meal_type_other2,meal_type_other3,meal_type_other4,pcs_mean_other1,pcs_mean_other2,pcs_mean_other3,pcs_mean_other4
0,-0.781831,0.62349,0.201299,0.97953,-2.449294e-16,1.0,che,115.521553,vegan,102.692308,89.854201,223.719787,0.0,0.0,vegan,fish,0,0,144.500000,240.516129,0.0,0.0
1,-0.781831,0.62349,0.201299,0.97953,-2.449294e-16,1.0,che,115.521553,vegan,102.692308,223.719787,89.854201,0.0,0.0,fish,vegan,0,0,240.516129,144.500000,0.0,0.0
2,-0.781831,0.62349,0.201299,0.97953,-2.449294e-16,1.0,che,89.854201,vegan,144.500000,115.521553,223.719787,0.0,0.0,vegan,fish,0,0,102.692308,240.516129,0.0,0.0
3,-0.781831,0.62349,0.201299,0.97953,-2.449294e-16,1.0,che,89.854201,vegan,144.500000,223.719787,115.521553,0.0,0.0,fish,vegan,0,0,240.516129,102.692308,0.0,0.0
4,-0.781831,0.62349,0.201299,0.97953,-2.449294e-16,1.0,che,223.719787,fish,240.516129,115.521553,89.854201,0.0,0.0,vegan,vegan,0,0,102.692308,144.500000,0.0,0.0


In [21]:
feat['pcs_pred'] = reg.predict(X)

feat.groupby('meal_id')['pcs_pred'].mean()

# feat.head()

meal_id
1307       173.578339
7614       187.927322
9500047     98.816139
Name: pcs_pred, dtype: float32